In [46]:
!pip install pyarrow
!pip install fastparquet

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.5 MB 1.1 MB/s eta 0:00:25
   - -------------------------------------- 0.8/27.5 MB 1.1 MB/s eta 0:00:25
   - -------------------------------------- 1.3/27.5 MB 1.3 MB/s eta 0:00:21
   -- ------------------------------------- 1.8/27.5 MB 1.5 MB/s eta 0:00:18
   --- ------------------------------------ 2.1/27.5 MB 1.5 MB/s eta 0:00:17
   --- ------------------------------------ 2.6/27.5 MB 1.7 MB/s eta 0:00:16
   ---- ----------------------------------- 3.1/27.5 MB 1.7 MB/s eta 0:00:15
   ---- ----------------------------------- 3.4/27.5 MB 1.7 MB/s eta 0:00:15
   ----- ---------------------------------- 3.7/27.5 MB 1.6 MB/s eta 0:00:15
   ----- ---------------------------------- 3.9/27.5 MB 1.7 MB/s eta 0:00:15
   ------ ----------

In [36]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
import pyarrow
import fastparquet
import pandas as pd

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/20/26 23:03:13] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=370954;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=954142;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


# Importar nodos

In [37]:
# 2. Importar todas las funciones del archivo nodes.py
import central_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))


['Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'central_preprocessing_calaca', 'central_preprocessing_estaca', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata']


In [38]:
# Muestra la ayuda formateada
help(nodes.clean_column_objects)

# O simplemente imprime el texto de la descripción
print(nodes.clean_column_objects.__doc__)

Help on function clean_column_objects in module central_perm_flow.pipelines.data_processing.nodes:

clean_column_objects(df: pandas.DataFrame, email_cols: list = None) -> pandas.DataFrame
    Limpia y estandariza columnas de tipo objeto en el DataFrame.

    Aplica conversión a minúsculas, eliminación de espacios y normalización de
    caracteres. Permite identificar columnas de correo para proteger su formato
    evitando la eliminación de '@' y '.'.

    Args:
        df (pd.DataFrame): DataFrame original con columnas de texto a procesar.
        email_cols (list, optional): Lista de nombres de columnas que deben
            tratarse como correos electrónicos. Por defecto es None.

    Returns:
        pd.DataFrame: DataFrame con las columnas de tipo objeto normalizadas.


Limpia y estandariza columnas de tipo objeto en el DataFrame.

Aplica conversión a minúsculas, eliminación de espacios y normalización de 
caracteres. Permite identificar columnas de correo para proteger su formato

# Fuentes de Información

## Base de Estados de los Estudiantes

In [39]:
# Base de central con fechas de bajas, fechas de graduación
central_caracterizacion = catalog.load('central_caracterizacion')

[04/20/26 23:03:18] INFO     Loading data from central_caracterizacion (CSVDataset)...         ]8;id=589357;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=970284;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

### Parámetros

In [42]:
central_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
central_col_emails = ['correo','correo_electronico_del_contacto_emergencia']
central_col_dd = ['identidad']
central_col_sort = ['identidad']

In [ ]:
# Limpieza de la base de operaciones
central_caracterizacion = nodes.clean_column_names(central_caracterizacion)
central_caracterizacion = nodes.convert_all_standardized_dates(central_caracterizacion,central_col_fechas)
central_caracterizacion['identidad'] = pd.to_numeric(central_caracterizacion['identidad'], errors='coerce')
central_caracterizacion = nodes.clean_column_objects(central_caracterizacion,central_col_emails)
central_caracterizacion_sd, central_caracterizacion_cd  = nodes.check_and_export_duplicates(central_caracterizacion, subset=central_col_dd, col_ordenar = central_col_sort)
central_caracterizacion



## Calendario Académico

In [44]:
# Calendario Académico
central_estados_calac = catalog.load("central_estados_calac")

[04/20/26 23:03:40] INFO     Loading data from central_estados_calac (ParquetDataset)...       ]8;id=681645;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=768378;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

### Parámetros

In [45]:
central_caracterizacion = central_estados_calac.merge(central_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])

In [47]:
central_caracterizacion.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel_x',
       ...
       'complemento', 'titulo_profesional_pregrado', 'ano_de_graduacion',
       'universidad_que_titula', '_ha_cursado_posgrados_adicionales_',
       'titulo_obtenido', 'fecha_graduacion',
       'universidad_que_otorga_el_titulo',
       '_ha_cursado_posgrados_en_la_universidad_central_', '_cual_'],
      dtype='str', length=124)

In [88]:
# limpieza del calendario académico
central_calendario = nodes.clean_column_names(central_calendario)
central_calendario = nodes.convert_all_standardized_dates(central_calendario, date_columns =  central_calaca_col_fechas)
central_calendario = nodes.clean_column_objects(central_calendario)

#Fecha de ingreso: la fecha de inicio del primer periodo académico en el que el estudiante estuvo activo
central_calendario[central_calaca_col_fechaingreso] = (
    central_calendario
    .groupby(central_calaca_col_cohorte_inicial)[central_calaca_col_fecha_inicial_semana]
    .transform('min')
)

# Ordenar el DataFrame por cohorte_inicial, cohorte_actual, bloque y semana
central_calendario_extendido = central_calendario.sort_values(by=central_calaca_col_sort).copy()
# Crear un contador de semanas acumuladas por cada cohorte (fecha_ingreso)
central_calendario_extendido['semana_acumulada'] = central_calendario_extendido.groupby(central_calaca_col_fechaingreso).cumcount() + 1

# 3. Calcular el mes corrido
# Usamos (semana - 1) // 4 + 1 para que las semanas 1, 2, 3 y 4 pertenezcan al Mes 1
central_calendario_extendido['month'] = (central_calendario_extendido['semana_acumulada'] - 1) // 4 + 1

# Formatear el mes corrido como 'm1', 'm2', etc.
central_calendario_extendido['mes_label'] = 'm' + central_calendario_extendido['month'].astype(str)
# Ordenar por fecha de ingreso, periodo inicial, bloque y semana
central_calendario_extendido = central_calendario_extendido.sort_values(by=central_calaca_col_sort).reset_index(drop=True)

# Shift para obtener la fecha de inicio del siguiente bloque
#grupos = [central_calaca_col_cohorte_inicial]
central_calendario_extendido['shifted_fecha_inicio'] = central_calendario_extendido.groupby(central_calaca_col_cohorte_inicial)[central_calaca_col_fecha_inicial_semana].shift(-1)
# Rellenar con la útlima fecha de fin para los casos donde no hay siguiente bloque
central_calendario_extendido['shifted_fecha_inicio'] = (
    central_calendario_extendido['shifted_fecha_inicio'].fillna(
        central_calendario_extendido.groupby(central_calaca_col_cohorte_inicial)['fecha_fin'].transform('max') 
    )
)

# Aplicamos de forma vectorizada (mucho más rápido)
central_calendario_extendido['student_journey'] = np.select(condiciones, central_calalaca_student_journey, default='unknown')

[04/18/26 17:25:22] WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=640561;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=36449;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:104: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               # 3. Convertir la columna date_column a datetime, manejando tanto                   
                             strings como números de serie de Excel                                                
                                                                                                                   

                    WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=465387;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=62636;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:104: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               # 3. Convertir la columna date_column a datetime, manejando tanto                   
                             strings como números de serie de Excel                                                
                                                                                                                   

In [89]:
import fastparquet

In [90]:
central_calendario_extendido = catalog.load('central_calendario_extendido')

                    INFO     Loading data from central_calendario_extendido                    ]8;id=511611;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=794470;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [91]:
central_estaca_sd = catalog.load('central_estaca_sd')

[04/18/26 17:25:52] INFO     Loading data from central_estaca_sd (ParquetDataset)...           ]8;id=233579;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=932516;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [94]:
central_estaca_sd['gi'].sum()

np.int64(145)

In [93]:
central_estaca_sd['di'].sum()

np.int64(186)